In [ ]:
import torch
from torch.utils import data
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import math

from datasets import Transforms, ImageNet, Places365, ArtPlaces
from models import CLIP, DINO_v2, CombinedModel, DINO_CLIP
from plot_utils import plot_points, plot_points_in_multiple_subplots

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if device.type == "cuda":
    print(torch.cuda.get_device_name())

In [ ]:
IMAGE_NUM = 1000
BATCH_SIZE = 128
FEATURE_EXTRACTION_METHOD = "pca"
PLOT_EACH_MODEL = False

DINO_CLIP_WEIGHTS = r"C:\Users\mariu\Documents\Studium\Masterprojekt\Gewichte\dino_clip\20260124_194812\state_dict_50.pt"

CLASSES_IMAGENET = [
    "tabby",
    "tiger cat",
    "Persian cat",
    "Egyptian cat",
    "cougar",
    "lynx",
    "German shepherd",
    "French bulldog",
    "Eskimo dog",
    "brown bear",
    "tractor",
    "warplane",
    "passenger car"
]

CLASSES_PLACES365 = [
    "amphitheater",
    "castle",
    "palace",
    "aqueduct",
    "alley",
    "highway",
    "beer_hall",
    "chalet",
    "botanical_garden",
    "beach",
    "mountain",
    "canyon",
]

CLASSES_ARTPLACES = [
    "amphitheater",
    "castle",
    "palace",
    "aqueduct",
    "alley",
    "highway",
    "beer_hall",
    "chalet",
    "botanical_garden",
    "beach",
    "mountain",
    "canyon",
]

In [ ]:
clip_model = CLIP(transform=Transforms.CLIP.value)
for p in clip_model.parameters():
    p.requires_grad = False
clip_model = clip_model.to(device)
clip_model = clip_model.eval()

dino_model = DINO_v2(transform=Transforms.DINO_v2.value)
for p in dino_model.parameters():
    p.requires_grad = False
dino_model = dino_model.to(device)
dino_model = dino_model.eval()

combined_model = CombinedModel(clip_transform=Transforms.CLIP.value, dino_transform=Transforms.DINO_v2.value)
for p in combined_model.parameters():
    p.requires_grad = False
combined_model = combined_model.to(device)
combined_model = combined_model.eval()

dino_clip = DINO_CLIP(transform=Transforms.DINO_CLIP.value, clip_transform=Transforms.CLIP.value, dino_transform=Transforms.DINO_v2.value)
for p in dino_clip.parameters():
    p.requires_grad = False
dino_clip = dino_clip.to(device)
dino_clip = dino_clip.eval()
dino_clip.load_state_dict(torch.load(DINO_CLIP_WEIGHTS))

In [ ]:
imagenet = ImageNet(root=r"C:\Users\mariu\Documents\Development\Datasets\imagenet-object-localization-challenge")
imagenet_subset = imagenet.getSubset(IMAGE_NUM, CLASSES_IMAGENET)
imagenet_dataloader = data.DataLoader(imagenet_subset, batch_size=BATCH_SIZE)

places365 = Places365(root=r"C:\Users\mariu\Documents\Studium\Praktikum\Places365_Evaluation_Subset")
places365_subset = places365.getSubset(IMAGE_NUM, CLASSES_PLACES365)
places365_dataloader = data.DataLoader(places365_subset, batch_size=BATCH_SIZE)

artplaces = ArtPlaces(root=r"C:\Users\mariu\Documents\Development\Datasets\ArtPlaces_13371280")
artplaces_subset = artplaces.getSubset(IMAGE_NUM, CLASSES_ARTPLACES)
artplaces_dataloader = data.DataLoader(artplaces_subset, batch_size=BATCH_SIZE)

In [ ]:
def extract_features(vectors, n_components=2):
    vectors_numpy = torch.flatten(vectors, start_dim=1).numpy()

    match FEATURE_EXTRACTION_METHOD:
        case "pca":
            fe = PCA(n_components=n_components)
            fe.fit(vectors_numpy)
            features = fe.transform(vectors_numpy)
        case "t-sne":
            fe = TSNE(n_components=n_components)
            features = fe.fit_transform(vectors_numpy)
    return features

In [ ]:
def compute_vecotrs(model, dataset, dataloader, classes, colors):
    images, labels = [],[]
    for i, l in dataloader:
        images.append(model(torch.clone(i).to(device)).cpu())
        labels.append(l)
    images = torch.cat(images)
    labels = torch.cat(labels)
    features_numpy = extract_features(images)
    labels_numpy = labels.numpy()

    legend = {}
    color_dict = {}
    for i in torch.unique(labels):
        legend[i.item()] = dataset.idx_to_class[i.item()]
        if colors is not None:
            color_dict[i.item()] = colors[classes.index(legend[i.item()])]
        else:
            color_dict = None
    
    return features_numpy, labels_numpy, legend, color_dict

In [ ]:
%matplotlib qt

features_numpy_list = []
labels_numpy_list = []
legend_list = []
color_dict_list = []
title_list = []

for model in [clip_model, dino_model, combined_model, dino_clip]:
    print(f"Processing model: {model.__class__.__name__}")

    for dataset, dataset_dataloader, classes in zip([imagenet, places365, artplaces], [imagenet_dataloader, places365_dataloader, artplaces_dataloader], [CLASSES_IMAGENET, CLASSES_PLACES365, CLASSES_ARTPLACES]):
        features_numpy, labels_numpy, legend, color_dict = compute_vecotrs(model, dataset, dataset_dataloader, classes, None)

        # title = f"{model_info['name']} - {dataset}"
        if PLOT_EACH_MODEL:
            plot_points(features_numpy, labels_numpy, legend=legend, title=None, colors=color_dict, figsize=(15, 10), label_cols=4, fontsize=25)

        features_numpy_list.append(features_numpy)
        labels_numpy_list.append(labels_numpy)
        legend_list.append(legend)
        color_dict_list.append(color_dict)
        title_list.append(f"{model.__class__.__name__} - {dataset.__class__.__name__}")

In [ ]:
cols = 3
rows = math.ceil(len(features_numpy_list) / cols)

In [ ]:
plot_points_in_multiple_subplots(
    points=features_numpy_list,
    labels=labels_numpy_list,
    legend=legend_list,
    subplot_titles=title_list,
    nrows=rows,
    ncols=cols,
    colors=color_dict_list,
    label_cols=4,
    fontsize=10
)

In [ ]:
plot_points_in_multiple_subplots(
    points=[f[0:len(f):2] for f in features_numpy_list],
    labels=[l[0:len(l):2] for l in labels_numpy_list],
    legend=legend_list,
    subplot_titles=title_list,
    nrows=rows,
    ncols=cols,
    colors=color_dict_list,
    label_cols=4,
    fontsize=10
)